# 실험 2: Classifier guidance를 붙인 Diffusion 오버샘플링

## 0. 실행 설정 / 라이브러리

In [ ]:
import os
import subprocess
import sys
import time
from pathlib import Path

RUN_SEEDS = list(range(30))        # 돌릴 seed
STEPS = [10]                       # 샘플링 스텝 수
SCALES = [0.5, 1.0, 2.0]           # guidance scale s
GPU_ID = None                      # None이면 가장 한가한 GPU를 자동 선택. 고정하려면 "0" / "1"처럼 지정
RESULTS_CSV = "exp2_classifier_guidance.csv"   # 항상 이어서 실행한다: 결과가 있는 (스텝, s, seed)는 건너뛰고 CSV는 뒤에 추가한다
NOISY_CLF_EPOCHS = 100             # 노이즈 분류기 학습 epoch (1 epoch = 24 step)
EVAL_EPOCHS = 100                  # 평가 분류기 학습 epoch (exp1과 같은 100)
POOL_PER_CLASS = 800 - 80          # 클래스당 fake 수 (exp1과 같은 720)

if GPU_ID is None:
    out = subprocess.check_output(
        ["nvidia-smi", "--query-gpu=index,memory.used,utilization.gpu", "--format=csv,noheader,nounits"]).decode()
    rows = [tuple(int(v) for v in l.split(",")) for l in out.strip().splitlines()]
    GPU_ID = str(min(rows, key=lambda r: (r[1], r[2]))[0])
os.environ["CUDA_VISIBLE_DEVICES"] = GPU_ID
print("사용 GPU:", GPU_ID)

In [ ]:
HERE = Path.cwd().resolve()
ROOT = HERE.parent  # 8_classifier_guidance/ (models.py, utils.py, diffusion.py, data.py)
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import transforms as T
from torchvision.datasets import ImageFolder
from tqdm.auto import tqdm

from models import Unet, Classifier, NoisyClassifier
from utils import set_seed, DEVICE, evaluate_classifier, metrics_to_rows, append_result_rows
from diffusion import GaussianDiffusion
from data import build_imbalanced_group_split, crop_group, CachedDataset, CachedSubset, draw_from_pool

## 1. 데이터 / 설정

In [ ]:
image_size, channels, batch_size, num_classes = 64, 3, 40, 3
data_root = os.environ.get("DATA_ROOT", "../../data/AM_image_data_3split")
CLASS_NAMES = ["Normal", "Underfill_50FR", "Underfill_Fan"]
train_counts = {0: 800, 1: 80, 2: 80}
test_counts = {0: 100, 1: 100, 2: 100}
timesteps = 1000
CLASSIFIER_LR, CLASSIFIER_BETAS = 2e-4, (0.5, 0.9)
NOISY_CLF_LR = 3e-4
GEN_BATCH = 360             # 한 번의 샘플링 호출로 생성할 장 수
EXP1_DIR = ROOT / "exp1_oversampling"   # 같은 날짜 폴더의 exp1(오버샘플링)가 학습해 둔 diffusion 체크포인트와 비교 기준 결과를 쓴다
DIFFUSION_EMA_PATH = str(EXP1_DIR / "checkpoints" / "exp1_ddpm1000_diffusion_ema_seed{seed}.pt")  # exp1에서 학습한 사전학습 모델

transform = T.Compose([T.Resize((image_size, image_size)), T.ToTensor(), T.Lambda(lambda t: t * 2 - 1)])
full_dataset = ImageFolder(root=data_root, transform=transform)
assert full_dataset.classes == CLASS_NAMES
full_cached = CachedDataset(full_dataset)
ddpm = GaussianDiffusion(timesteps=timesteps, image_size=image_size, channels=channels)

train_dataset = test_dataset = train_loader = test_loader = None

## 2. 함수 정의

In [ ]:
def set_split(seed):
    global train_dataset, test_dataset, train_loader, test_loader
    tr, te = build_imbalanced_group_split(full_dataset, train_counts, test_counts, seed=seed)
    leaked = ({crop_group(full_dataset.samples[i][0]) for i in tr} & {crop_group(full_dataset.samples[i][0]) for i in te})
    assert not leaked, f"같은 원본이 train/test에 모두 있음: {sorted(leaked)[:5]}"
    train_dataset, test_dataset = CachedSubset(full_cached, tr), CachedSubset(full_cached, te)
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, drop_last=True, pin_memory=(DEVICE == "cuda"))
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, pin_memory=(DEVICE == "cuda"))


def train_noisy_classifier(seed):
    clf = NoisyClassifier(num_classes=num_classes, img_channels=channels).to(DEVICE)
    labels = torch.as_tensor([train_dataset[i][1] for i in range(len(train_dataset))])
    counts = torch.bincount(labels, minlength=num_classes).float()
    weight = (counts.sum() / (num_classes * counts)).to(DEVICE)
    set_seed(seed)
    opt = torch.optim.Adam(clf.parameters(), lr=NOISY_CLF_LR)
    loss_fn = nn.CrossEntropyLoss(weight=weight)
    clf.train()
    total, it, losses, t0 = len(train_loader) * NOISY_CLF_EPOCHS, iter(train_loader), [], time.time()
    for step in tqdm(range(total), desc=f"[NoisyClf seed{seed}]", leave=False):
        try:
            x, y = next(it)
        except StopIteration:
            it = iter(train_loader)
            x, y = next(it)
        x, y = x.to(DEVICE), y.to(DEVICE)
        t = torch.randint(0, timesteps, (x.size(0),), device=DEVICE)
        loss = loss_fn(clf(ddpm.q_sample(x, t), t), y)
        opt.zero_grad(); loss.backward(); opt.step()
        losses.append(loss.item())
        if (step + 1) % max(1, total // 5) == 0:
            tqdm.write(f"[NoisyClf seed{seed}] step {step+1}/{total} CE={np.mean(losses[-100:]):.4f} elapsed={time.time()-t0:.1f}s")
    clf.eval()
    return clf


@torch.no_grad()
def noisy_clf_report(clf, seed):
    res = {}
    for t_val in (50, 250, 500, 750, 950):
        correct = n = 0
        for x, y in test_loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            t = torch.full((x.size(0),), t_val, device=DEVICE, dtype=torch.long)
            correct += (clf(ddpm.q_sample(x, t), t).argmax(1) == y).sum().item()
            n += x.size(0)
        res[t_val] = correct / n
    print(f"[NoisyClf seed{seed}] test 정확도(노이즈 t별): " + ", ".join(f"t={k}: {v:.2f}" for k, v in res.items()))


def build_guided_pool(model, noisy_clf, steps, scale, minority_classes=(1, 2)):
    pool = {}
    for c in minority_classes:
        xs, n_left = [], POOL_PER_CLASS
        while n_left > 0:
            n = min(GEN_BATCH, n_left)
            y = torch.full((n,), c, dtype=torch.long, device=DEVICE)
            xs.append(ddpm.sample_ddpm_respaced_cg(model, noisy_clf, y, DEVICE, steps=steps, scale=scale))
            n_left -= n
        pool[c] = torch.cat(xs)
    return pool


def train_classifier_pool(seed, pool, total_steps, tag):
    set_seed(seed)
    clf = Classifier(num_classes=num_classes, img_channels=channels, image_size=image_size).to(DEVICE)
    opt = torch.optim.Adam(clf.parameters(), lr=CLASSIFIER_LR, betas=CLASSIFIER_BETAS)
    loss_fn = nn.CrossEntropyLoss()
    clf.train()
    it, losses, t0 = iter(train_loader), [], time.time()
    for step in tqdm(range(total_steps), desc=f"[{tag} seed{seed}]", leave=False):
        try:
            x, y = next(it)
        except StopIteration:
            it = iter(train_loader)
            x, y = next(it)
        x, y = x.to(DEVICE), y.to(DEVICE)
        xf, yf = draw_from_pool(pool, y, DEVICE)
        if xf is not None:
            x, y = torch.cat([x, xf]), torch.cat([y, yf])
        loss = loss_fn(clf(x), y)
        opt.zero_grad(); loss.backward(); opt.step()
        losses.append(loss.item())
        if (step + 1) % max(1, total_steps // 5) == 0:
            tqdm.write(f"[{tag} seed{seed}] step {step+1}/{total_steps} Cls={np.mean(losses[-50:]):.4f} elapsed={time.time()-t0:.1f}s")
    return clf

## 실험 2: Classifier guidance 오버샘플링 & 평가

In [ ]:
done = set()
if os.path.exists(RESULTS_CSV):
    _prev = pd.read_csv(RESULTS_CSV)
    done = set(zip(_prev["method"], _prev["seed"]))

for seed in RUN_SEEDS:
    set_split(seed)
    eval_steps = len(train_loader) * EVAL_EPOCHS
    ema_path = DIFFUSION_EMA_PATH.format(seed=seed)
    if not os.path.exists(ema_path):
        print(f"[seed{seed}] 체크포인트 없음 → 건너뜀: {ema_path}")
        continue
    model = Unet(dim=32, channels=channels, num_classes=num_classes).to(DEVICE)
    model.load_state_dict(torch.load(ema_path, map_location=DEVICE))
    model.eval()
    print(f"\n===== Classifier guidance oversampling | Seed {seed} (사전학습 EMA 로드: {ema_path}) =====")
    noisy_clf = train_noisy_classifier(seed)
    noisy_clf_report(noisy_clf, seed)
    for steps in STEPS:
        for scale in SCALES:
            method = f"2_classifier_guidance_ddpm1000_step{steps}_s{scale:g}"
            if (method, seed) in done:
                print(f"[{method} seed{seed}] 결과 있음 → 건너뜀"); continue
            t0 = time.time()
            set_seed(seed)   # 초기 노이즈가 s마다 같다
            pool = build_guided_pool(model, noisy_clf, steps, scale)
            print(f"[{method} seed{seed}] 풀 생성 {time.time()-t0:.1f}초")
            clf = train_classifier_pool(seed, pool, eval_steps, tag=f"s{scale:g}-step{steps}")
            metrics = evaluate_classifier(clf, test_loader, DEVICE, num_classes, CLASS_NAMES)
            rows = metrics_to_rows(metrics, seed, method)
            append_result_rows(rows, RESULTS_CSV)
            print(f"[{method} seed{seed}] 총 {time.time()-t0:.1f}초")
            del pool

In [ ]:
if os.path.exists(RESULTS_CSV) and os.path.getsize(RESULTS_CSV) > 0:
    df = pd.read_csv(RESULTS_CSV).drop_duplicates(subset=["method", "seed", "class"], keep="last")
    df1 = pd.read_csv(EXP1_DIR / "exp1_oversampling.csv")
    m = df[df["class"].astype(str) == "macro"]
    m1 = df1[df1["class"].astype(str) == "macro"]
    seeds_done = sorted(m["seed"].unique())
    print(f"\n===== macro F1 (seed {seeds_done}) =====")
    for st in sorted({int(x.split("step")[1].split("_")[0]) for x in m["method"]}):
        ref = m1[(m1["method"] == f"1_diffusion_oversampling_ddpm1000_step{st}") & m1["seed"].isin(seeds_done)]
        print(f"exp1 오버샘플링 (가이던스 없음, {st}스텝): {ref['f1'].mean():.4f} (n={len(ref)})")
        for meth in sorted(x for x in m["method"].unique() if f"step{st}_" in x):
            mm = m[m["method"] == meth]
            print(f"  {meth}: {mm['f1'].mean():.4f} (n={len(mm)})")